# 面向长对话的贝叶斯自适应上下文选择 · Release研究任务工作本
**ARIN7101 · release-2.1 scaffold · 2026-11-05**

研究问题：在受控合成长对话中，利用问题特征的贝叶斯策略选择能否改善正确性与输入成本的取舍？

**这是任务脚手架，研究实现和结果评估由成员自行完成。** 这里只提供公共路径、实验设计草案、接口骨架与任务提示。研究函数均以NotImplementedError留空；第0节会检查环境、建立统一目录并保存环境记录；M1–M5仍只定义骨架，不生成研究数据、统计或图，初始化成功不代表任务完成。

先读[README](README.md)和[协作说明](docs/TEAM_WORKFLOW.md)。每个人须交付自己的代码、自己设计的验证、实际观察与局限和可追踪产物。参考完整本只作可行性背景，release不从中导入或执行实现；旧数值、图表与结论不属于本组已完成的研究。

| 成员 | 必须自己研究、编码的范围 | cell ID区间 | 接收人 |
|---|---|---|---|
| [M1](#member-m1) | 数据生成、公开信息边界、独立真值核查 | m1-* | M2/M4 |
| [M2](#member-m2) | 四策略、回答器、成本、评价及反例 | m2-* | M1/M3/M4 |
| [M3](#member-m3) | 特征、TS/CTS、后验数值验证、回放 | m3-* | M4 |
| [M4](#member-m4) | 开发选择、冻结、实验调度、统计与归档 | m4-* | M3复核/M5 |
| [M5](#member-m5) | 读取校验、可视化编码、结果解释与汇报 | m5-* | 全组 |


## 0 · 环境配置及路径约定
**本节代码已写好，全员首次下载仓库后直接按顺序运行本节两个代码单元。** 它负责依赖检查、公共导入、根目录定位、统一目录初始化、环境记录及安全保存工具。之后各成员使用这里的路径变量；无需自行拼接绝对路径。

### 0.1 · 首次准备与环境检查
建议Python 3.11或3.12。在仓库根目录创建虚拟环境并安装Jupyter（首次尚无法打开notebook时执行）：

    python -m venv .venv
    # Windows PowerShell：.\.venv\Scripts\Activate.ps1
    # macOS/Linux：source .venv/bin/activate
    python -m pip install -r requirements.txt
    python -m ipykernel install --user --name arin7101 --display-name "Python 3 (ARIN7101)"
    python -m jupyter lab

选择上述内核。下一单元使用当前内核的Python检查requirements；若依赖缺失或版本不符，默认自动安装，并要求重启内核后重跑，以免混用已加载的旧包。无需GPU、API密钥或外部数据。

PROJECT_ROOT默认自动从当前目录向上寻找仓库；若从仓库外启动内核，将它填为本机仓库根路径。不要把自己的绝对路径提交Git。AUTO_INSTALL_MISSING可改为False，只检查并给出安装提示。


In [ ]:
# 0.1：全员直接运行。只处理环境与公共导入。
import sys
import os
import re
import json
import platform
import subprocess
from pathlib import Path
from datetime import datetime
from importlib import metadata
from packaging.requirements import Requirement
from IPython.display import display

PROJECT_ROOT = None  # 通常无需填写；仓库外启动时可填本机路径
AUTO_INSTALL_MISSING = True

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("本项目统一使用Python 3.11或3.12；请切换对应Jupyter内核。")
_candidates = ([Path(PROJECT_ROOT).expanduser()] if PROJECT_ROOT
               else [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in _candidates
             if (p / "ARIN7101_Context_Bandit_Release_Notebook.ipynb").is_file()
             and (p / "requirements.txt").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("未定位仓库，请填写PROJECT_ROOT或从仓库内启动Jupyter。")

_requirements = []
for _line in (ROOT / "requirements.txt").read_text(encoding="utf-8-sig").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if _line:
        _req = Requirement(_line)
        if _req.marker is None or _req.marker.evaluate():
            _requirements.append(_req)

_problems = []
for _req in _requirements:
    try:
        _version = metadata.version(_req.name)
    except metadata.PackageNotFoundError:
        _problems.append(f"{_req.name} 未安装")
    else:
        if _version not in _req.specifier:
            _problems.append(f"{_req.name}={_version}，要求 {_req.specifier}")
if _problems:
    print("需要处理的依赖：\n- " + "\n- ".join(_problems))
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError(f'请用当前内核Python安装依赖："{sys.executable}" -m pip install -r "{ROOT / "requirements.txt"}"')
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")],
                   check=True)
    raise RuntimeError("依赖已安装。请Restart Kernel，重新运行第0节，避免混用已加载的旧包。")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "executable": sys.executable,
    "packages": {_req.name: metadata.version(_req.name) for _req in _requirements},
}
print("环境检查通过")
print("项目根目录：", ROOT)
print("当前内核：", sys.executable)
display(pd.DataFrame(list(ENVIRONMENT["packages"].items()), columns=["package", "version"]))


### 0.2 · 初始化工作目录和公共路径
运行下一单元，默认建立本机练习目录outputs/development/local_setup及其六个子目录，并写入config/environment.json。已有文件与人工记录不覆盖。重复执行同一个development/demo配置可复用目录；正式run必须是新目录。

**每次实际实验开始前修改MODE和RUN_ID**，例如development / m1_trial01，或demo / team_integration01。成员交接必须明确run路径；同名run只有在配置、输入与来源一致时才能合用。正式运行由M4完成审查与冻结后初始化，初始化不等于获得正式执行资格。

| 路径/变量 | 用途与负责成员 | 本节行为 | Git |
|---|---|---|---|
| ROOT | 仓库根目录，自动定位 | 查找，不改工作目录 | — |
| TEAM_DIR = team_workspace/ | 姓名、进度、接口决策、正式尝试登记；全员/M4 | 确保目录存在，不改人工记录 | 提交 |
| DELIVERABLES_DIR = deliverables/ | 个人研究说明、最终slides、提交清单；全员 | 确保目录存在 | 提交小文件，视频用链接 |
| DOCS_DIR / REFERENCE_DIR | 协作规范、原计划、历史参考 | 仅定位/确保目录存在 | 提交 |
| OUTPUTS_DIR = outputs/ | 自动产物总目录 | 建立development/demo/formal三个分区 | 忽略 |
| RUN_DIR = outputs/MODE/RUN_ID/ | 本次开发/实验的唯一根目录 | 初始化；formal禁止复用已有目录 | 忽略 |
| DATA_DIR = RUN_DIR/data/ | 生成对话与原始数据；M1/M4 | 创建空目录 | 忽略 |
| TABLES_DIR = RUN_DIR/tables/ | 开发评价、回放、统计表、绘图数据；M2–M5 | 创建空目录 | 忽略 |
| AUDIT_DIR = RUN_DIR/audit/ | 真值/策略/方法检查、十题审查与证据副本；M1–M4 | 创建空目录 | 忽略 |
| CONFIG_DIR = RUN_DIR/config/ | 环境记录、冻结配置、代码快照；公共设施/M4 | 写环境记录，其余由成员完成 | 忽略 |
| FIGURES_DIR = RUN_DIR/figures/ | 自己编写代码生成的PNG/SVG；M5 | 创建空目录 | 忽略 |
| REPORTS_DIR = RUN_DIR/reports/ | 结果交接、逐图解读、图表清单；M4/M5 | 创建空目录 | 忽略 |
| RUN_DIR/manifest.json | 整个实验的来源与文件校验清单；M4 | 不生成，留给M4实现 | 忽略 |

**统一使用方式：**
- 自己实现的函数接收RUN_DIR；单文件用artifact_path(MODE, RUN_ID, area, filename)取路径。
- 已提供save_json(path, value)和save_csv(path, dataframe)；它们限定outputs内部、UTF-8编码，拒绝覆盖已有文件。不再让每人重复写路径/通用保存代码。
- 绘图、压缩对话等专用导出仍由成员实现，使用上述目录。JSON、CSV的内容与研究指标仍由成员计算。
- M5读取M4交付的明确路径作为SOURCE_RUN，不重新初始化/改写上游formal目录；可以另用自己的development run试绘。
- 本节只有环境信息，不生成研究配置冻结、数据、统计、图或成员签名。使用新RUN_ID保留修改前产物；不要以重跑初始化来清理旧结果。


In [ ]:
# 0.2：全员直接运行；做实际实验时改这两个值，不改下面的统一目录规则。
MODE = "development"     # development / demo / formal
RUN_ID = "m2_trial01"   # 正式实验由M4在冻结后使用全新ID

TEAM_DIR = ROOT / "team_workspace"
DELIVERABLES_DIR = ROOT / "deliverables"
DOCS_DIR = ROOT / "docs"
REFERENCE_DIR = ROOT / "reference"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUT_AREAS = ("data", "tables", "audit", "config", "figures", "reports")
MODES = ("development", "demo", "formal")

def _safe_component(value):
    if (not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", value)
            or value.endswith(".") or value.split(".", 1)[0].upper() in
            {"CON", "PRN", "AUX", "NUL", *[f"COM{i}" for i in range(1, 10)],
             *[f"LPT{i}" for i in range(1, 10)]}):
        raise ValueError("名称须以字母/数字开头，只含字母、数字、下划线、连字符和点；不可用系统保留名。")
    return value

def _run_path(mode, run_id):
    if mode not in MODES:
        raise ValueError(f"mode须为{MODES}")
    path = OUTPUTS_DIR / mode / _safe_component(run_id)
    if not path.resolve().is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("路径必须位于outputs内。")
    return path

def artifact_path(mode, run_id, area, filename):
    """返回统一文件路径；先使用init_run初始化该run，不在此处生成研究结果。"""
    if area not in OUTPUT_AREAS:
        raise ValueError(f"area须为{OUTPUT_AREAS}")
    path = _run_path(mode, run_id) / area / _safe_component(filename)
    if not path.resolve().is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("路径必须位于outputs内。")
    return path

def _writable_output(path):
    path = Path(path).resolve()
    if not path.is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("通用保存工具只写outputs；人工记录请本人编辑。")
    if not path.parent.is_dir():
        raise FileNotFoundError("输出目录不存在；请先init_run。")
    return path

def save_json(path, value):
    """统一UTF-8/严格JSON，拒绝覆盖；不代替研究计算或签名。"""
    payload = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
    with _writable_output(path).open("x", encoding="utf-8") as stream:
        stream.write(payload)

def save_csv(path, dataframe):
    """保存成员计算得到的DataFrame；统一UTF-8、无索引、拒绝覆盖。"""
    dataframe.to_csv(_writable_output(path), index=False, encoding="utf-8", mode="x")

def init_run(mode, run_id):
    """创建六个目录和环境记录；开发/demo可复用，formal禁止复用。"""
    run_dir = _run_path(mode, run_id)
    if mode == "formal" and run_dir.exists():
        raise FileExistsError("正式run目录已存在；请保留原目录，不能重跑初始化覆盖。")
    # 先检查既有环境，不改已有记录；环境变化应使用新的run_id。
    env_path = run_dir / "config" / "environment.json"
    if env_path.exists():
        existing = json.loads(env_path.read_text(encoding="utf-8"))
        if (existing.get("python"), existing.get("packages")) != (
                ENVIRONMENT["python"], ENVIRONMENT["packages"]):
            raise RuntimeError("本run已记录不同Python/依赖版本；请用新RUN_ID保留环境来源。")
    run_dir.mkdir(parents=True, exist_ok=(mode != "formal"))
    for area in OUTPUT_AREAS:
        directory = run_dir / area
        if not directory.resolve().is_relative_to(run_dir.resolve()):
            raise ValueError("子目录不能指向run以外的位置。")
        directory.mkdir(exist_ok=True)
    if not env_path.exists():
        save_json(env_path, {
            **ENVIRONMENT, "mode": mode, "run_id": run_id,
            "created_at": datetime.now().astimezone().isoformat(),
            "status": "environment_initialized_only",
        })
    return run_dir

for directory in (TEAM_DIR, DELIVERABLES_DIR, DOCS_DIR, REFERENCE_DIR, OUTPUTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
for mode in MODES:
    (OUTPUTS_DIR / mode).mkdir(exist_ok=True)

RUN_DIR = init_run(MODE, RUN_ID)
RUN_PATHS = {area: RUN_DIR / area for area in OUTPUT_AREAS}
DATA_DIR = RUN_PATHS["data"]
TABLES_DIR = RUN_PATHS["tables"]
AUDIT_DIR = RUN_PATHS["audit"]
CONFIG_DIR = RUN_PATHS["config"]
FIGURES_DIR = RUN_PATHS["figures"]
REPORTS_DIR = RUN_PATHS["reports"]

print("初始化完成；未生成研究结果。")
print("RUN_DIR =", RUN_DIR)
display(pd.DataFrame(
    [{"area": area, "relative_path": directory.relative_to(ROOT).as_posix(),
      "absolute_path": str(directory)} for area, directory in RUN_PATHS.items()]
))
print("环境记录：", CONFIG_DIR / "environment.json")

# 在成员完成计算后使用（这里只示范路径，不生成假的结果文件）：
# save_csv(artifact_path(MODE, RUN_ID, "tables", "development_outcomes.csv"), outcomes)
# save_json(artifact_path(MODE, RUN_ID, "audit", "truth_checks.json"), checks)
# M5使用明确的SOURCE_RUN读取M4交付物，不调用init_run去重建上游formal目录。


## 共同研究设计与成员探索空间
以下设计草案供M1–M5研究；与第0节的环境初始化分开，确认冻结由M4负责。

**原计划提供的设计起点，开发结束前由全组确认冻结：**
- 6个项目、4种属性，英文模板历史含focus、事实更新、旧版本与干扰；每对话300题。
- stable的近期题概率0.9；abrupt在第151题起从0.9变0.1。改变的是提问需求分布。
- recent_short：最近最多3条，预算128；recent_long：连续近期窗口，512；tfidf：原问题检索最多8个候选，256；resolved_tfidf：用公开历史消解指代/同义词后检索，256。
- U = Y − λC/Cref，λ=0.5、Cref=512。C是整个输入的正则长度代理，不是真实token费用；预算仅限制所选上下文。
- 开发每场景3条、demo每场景6条、正式每场景30条。TS/CTS算法seed为17/37，每对话重置。
- 主比较：CTS与各场景开发集选定的固定臂；TS为次比较；RuleRouter为模板规则参照。算法seed先在对话内平均，再按独立对话配对bootstrap，报告95%和两主比较校正97.5%区间。

**成员需要自行决定并研究：** 具体模板与版本生成细节、检索平局、数值优化、失败处理、测试案例与图型选择。记录理由、比较和局限，不能只复制完整本。对共同设计的修改须在正式评估前确认；测试结果之后的修改另标探索/修正。

**并行起步：** M1先做可追踪小历史；M2用自建历史；M3用自建四臂小表；M4设计统计手算案例；M5用明确标为fixture的虚构表研究绘图。示意数据不进入真实结果。

**接通顺序：** M1生成器 → M2评价器 → M1/M2共同审计 → M3回放 → M4集成与冻结 → 正式运行 → M5解释。未实现函数被调用时明确报错，不用空表/固定返回值掩盖未完成。


In [ ]:
ARMS = ("recent_short", "recent_long", "tfidf", "resolved_tfidf")
SCENARIOS = ("stable", "abrupt")
METHODS = ARMS + ("TS", "LaplaceCTS", "RuleRouter")
PROTOCOL_DRAFT = {
    "T": 300, "n_dev_per_scenario": 3, "n_demo_per_scenario": 6,
    "n_formal_per_scenario": 30, "dev_seed": 130000, "demo_seed": 730000,
    "formal_seed": None,  # M4与全组确认未使用的seed后填写
    "algorithm_seeds": [17, 37], "p_recent_stable": 0.9,
    "p_recent_abrupt_before": 0.9, "p_recent_abrupt_after": 0.1,
    "update_probability": 0.25, "initial_conflict_fraction": 0.25,
    "budgets": dict(zip(ARMS, [128, 512, 256, 256])),
    "lambda": 0.5, "cost_reference": 512, "refit_every": 5,
    "prior_precision": 1.0, "bootstrap_seed": 2026, "n_bootstrap": 2000,
}


### 共同接口
详细字段见[输出契约](docs/OUTPUT_CONTRACT.md)。骨架参数名是交接约定；模块内部实现由成员选择。
- episode：seed/scenario/history/probes；message：id/text；probe：t/question/end/gold/evidence/old_values/qtype/state。
- public_view仅含question/history；gold、隐藏题型、未选反馈只供评估器访问。
- 四臂表：seed,scenario,t,arm,question,y,cost,utility,coverage,outdated,construction_ms,qtype,state。
- 回放表：seed,scenario,t,method,alg_seed,arm,y,cost,utility,coverage,outdated。
- 对话指标：scenario,seed,method,y,cost,utility,coverage,outdated。
- 主效应表：scenario,method,baseline,n,effect,low,high,family_low,family_high。

消费者自行检查字段、唯一键、规模与来源。上游未完成时自行构造明确标为fixture的最小输入，不用完整参考结果填空。


<a id="member-m1"></a>
## M1 · 数据与问题定义
**你要编写生成器、公开视图和独立核查器，研究数据机制。任务区间m1-*。**
输入：共同设计；输出：episode、数据字典、开发数据、核查证据；M2/M4接收。

1. **先设计。** 用纸面例子定义focus/its、对象、属性、最新事实和过期值；决定多版本、复述、同义词如何生成，记录选择理由。
2. **先小后大编码。** 从少量题目的生成器扩展到两个场景与300题；保持当前题只能看历史前缀。
3. **独立核查。** 设计另一条证据追踪路径，不把生成器存的gold读出来就当证明；写复现、指代、更新、旧值、切换边界案例。
4. **交付与联调。** 先给M2微型episode和字典，再导出开发集。M2导出四臂审计后，你追踪10题真值，M2交叉复核。
5. **诊断与解释。** 检查实际题型/版本分布，说明是否人为偏向某种路由；发现问题在开发阶段修改，不预设策略一定胜出。

**验收方向（案例由你设计）：** seed可复现、id连续、无未来证据、真值可追溯、切换正确。接收人能从一条样例找到答案依据。
**输出：** data/development_dialogues.jsonl.gz；audit/data_dictionary.json、truth_checks.json；tables/development_data_profile.csv。使用明确的development run路径。


In [ ]:
def generate_episode(seed, scenario, cfg):
    """M1实现：按公共字段返回episode，从微型例子开始。"""
    raise NotImplementedError("M1：设计生成机制并实现")

def public_view(episode, probe):
    """M1实现：只返回question与当轮公开history。"""
    raise NotImplementedError("M1：实现公开信息边界")

def audit_truth(episode):
    """M1实现：独立追踪真值，返回可定位seed/t/evidence的核查记录。"""
    raise NotImplementedError("M1：设计独立核查")

def export_development_data(episodes, run_dir):
    """M1实现：按输出契约保存数据、字典与核查；run_dir为本次run根目录。"""
    raise NotImplementedError("M1：实现数据归档")


In [ ]:
# TODO M1：自行添加微型输入、断言、数据诊断和导出调用。
# 同seed复现；更新前后；its/显式对象；切换边界。
# 先写预期证据，再检查自己的实现；保留失败与修复，给M2验收。


### M1研究记录（本人填写）
哪些是设计约束，哪些是自己的选择？独立核查发现过什么错误？数据机制支持什么范围的结论？
代码/测试cell、证据run、M2/M4接收记录：待填写。


<a id="member-m2"></a>
## M2 · 策略、回答器与成本
**四臂选择、规则回答与评价代码由你编写。任务区间m2-*。**
输入：M1公开视图与episode；上游未完成时自建短历史。输出给M3/M4，审计材料交M1。

1. **定义公平比较。** 研究共同输入模板、计数规则、UNKNOWN与最新事实优先规则；四臂必须共享回答器与成本口径。
2. **逐臂实现。** 从连续窗口到TF–IDF，再实现仅用公开历史的指代消解和同义词处理；记录平局、超预算消息、focus锚点处理。
3. **写自己的测试。** 检查预算边界、空历史、未来隔离、时序、旧值；不可读取gold/隐藏题型。
4. **评价与研究。** 为每题每臂导出行，记录完整输入成本和候选构造耗时，研究accuracy/coverage/outdated的关系；结果由实验决定。
5. **交接。** 导出10题的公开历史、各臂上下文/回答/成本和评估真值；M1独立追踪，你交叉复核。先给M3单对话表，再给M4全部开发表。

**验收方向：** 预算只管上下文；C含共同指令、上下文、问题；id唯一且时序；完整/空历史行为有依据。至少三个自己发现或构造的边界案例及测试局限。
**输出：** tables/development_outcomes.csv、strategy_by_query.csv；audit/audit_examples.json、audit_review.csv、strategy_checks.json。人工审计不能自动填pass。


### M2 第一个里程碑 · 从短历史走通四臂评价

运行第0节和 `shared-protocol-draft` 后，按顺序运行下面各格即可；本例不调用尚未实现的M1生成器。
本节只使用明确标为 **fixture** 的9条消息、4个问题，不把它们写成正式开发集或实验结论。

| 步骤 | 阅读重点 | 可见结果 |
|---|---|---|
| A | 统一计数与输入模板 | 一段文字如何变成成本单位 |
| B | 事实、同义词、focus与回答器 | 全历史的手工答案对照 |
| C | 预算打包与四种策略 | 各臂选中消息、预算、TF–IDF排序 |
| D | 独立评价与成本分解 | 16行四臂表、完整输入示例 |
| E | 边界检查 | 预算、时序、空输入、未来隔离与指代反例 |

参考来源：[v1完整参考本](reference/ARIN7101_Context_Bandit_Release_v1_Complete_Reference.ipynb) 的
`data-generator` 中解析规则、`strategy-evaluation` 中窗口/检索/打包思路及 `shared-config` 中正则计数。
这里重新组织为可分步运行的模块，显式处理空词表、校验公开视图，并通过独立手写预期值检查样例。
与M1尚未联调：当前仅支持参考本的英文模板（`P数字`、`V数字`、四种属性）；不是通用自然语言回答器。


#### A · 统一计数与输入模板

使用 `\w+|[^\w\s]`：每个连续词串或独立标点计1单位，空白不计。
上下文预算只计算消息正文；总成本还包括共同指令、`Context:` / `Question:` 标签和问题。
四臂共用同一模板。这是长度代理，不是真实token、API费用或延迟。


In [ ]:
import re
from copy import deepcopy
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

M2_INSTRUCTION = (
    "Answer using only the context. Later facts override earlier facts. "
    "Resolve its using the latest focus. Return the value, or UNKNOWN if unsupported."
)


def count_input_units(text):
    """正则长度代理；例如 P1: V11. 被分为 P1 / : / V11 / .。"""
    return len(re.findall(r"\w+|[^\w\s]", text))


def m2_context_text(context):
    return "\n".join(message["text"] for message in context)


def m2_build_prompt(question, context):
    return M2_INSTRUCTION + "\nContext:\n" + m2_context_text(context) + "\nQuestion: " + question


sample_text = "P1: V11."
display(pd.DataFrame([{
    "text": sample_text,
    "regex_units": re.findall(r"\w+|[^\w\s]", sample_text),
    "count": count_input_units(sample_text),
}]))


#### B · 从公开文本识别对象、属性和最新事实

先把 `data source / coding language / evaluation measure / due date` 归一化为四种属性。
显式项目名优先；只有问题包含 `its` 时才使用**所给上下文**中的最新focus。
回答器依消息顺序更新事实，最后一个版本覆盖旧版本；缺对象、属性或事实都返回 `UNKNOWN`。
回答器不接收gold、evidence或完整episode。


In [ ]:
M2_ALIASES = {
    "dataset": "data source", "language": "coding language",
    "metric": "evaluation measure", "deadline": "due date",
}
M2_PROJECT = re.compile(r"\bP\d+\b", re.I)
M2_FOCUS = re.compile(r"\bFocus is project (P\d+)\b", re.I)
M2_FACT = re.compile(
    r"\bproject (P\d+): the (dataset|language|metric|deadline) is (V\d+)\b", re.I
)


def m2_normalize(text):
    for canonical, alias in M2_ALIASES.items():
        text = re.sub(r"\b" + re.escape(alias) + r"\b", canonical, text, flags=re.I)
    return text


def m2_parse_fact(text):
    match = M2_FACT.search(m2_normalize(text))
    if match is None:
        return None
    project, attribute, value = match.groups()
    return project.upper(), attribute.lower(), value.upper()


def m2_resolve_target(question, context):
    explicit = M2_PROJECT.search(question)
    project = explicit.group().upper() if explicit else None
    if project is None and re.search(r"\bits\b", question, re.I):
        for message in context:
            focus = M2_FOCUS.search(message["text"])
            if focus:
                project = focus.group(1).upper()
    normalized = m2_normalize(question)
    attribute = next((name for name in M2_ALIASES
                      if re.search(r"\b" + name + r"\b", normalized, re.I)), None)
    return project, attribute


def rule_reader(question, context):
    """只使用调用者给定的、按时间排列的上下文。"""
    target = m2_resolve_target(question, context)
    answer = "UNKNOWN"
    for message in context:
        fact = m2_parse_fact(message["text"])
        if fact and fact[:2] == target:
            answer = fact[2]
    return answer


In [ ]:
# 手工fixture：id 0/3/8 是P1的旧值、更新、同值复述；id 2/5 是focus切换。
M2_FIXTURE = {
    "seed": 42, "scenario": "fixture",
    "history": [
        {"id": 0, "text": "For project P1: the dataset is V10."},
        {"id": 1, "text": "For project P2: the language is V20."},
        {"id": 2, "text": "Focus is project P1. We are reviewing its latest requirements."},
        {"id": 3, "text": "For project P1: the data source is V11."},
        {"id": 4, "text": "The next meeting will review progress and remaining questions."},
        {"id": 5, "text": "Focus is project P2. We are reviewing its latest requirements."},
        {"id": 6, "text": "For project P2: the coding language is V21."},
        {"id": 7, "text": "The next meeting will review progress and remaining questions."},
        {"id": 8, "text": "For project P1: the data source is V11."},
    ],
    "probes": [
        {"t": 0, "question": "What is P1 current dataset?", "end": 3,
         "gold": "V10", "evidence": 0, "old_values": [], "qtype": "historical", "state": "A"},
        {"t": 1, "question": "What is its current data source?", "end": 5,
         "gold": "V11", "evidence": 3, "old_values": ["V10"], "qtype": "recent", "state": "A"},
        {"t": 2, "question": "What is P1 current dataset?", "end": 8,
         "gold": "V11", "evidence": 3, "old_values": ["V10"], "qtype": "historical", "state": "A"},
        {"t": 3, "question": "What is its current coding language?", "end": 9,
         "gold": "V21", "evidence": 6, "old_values": ["V20"], "qtype": "recent", "state": "A"},
    ],
}
M2_CFG = deepcopy(PROTOCOL_DRAFT)
display(pd.DataFrame(M2_FIXTURE["history"]))
display(pd.DataFrame(M2_FIXTURE["probes"])[
    ["t", "end", "question", "gold", "evidence", "old_values"]
])
print("end是Python切片的右开边界：t=0只能看id 0、1、2，不能看id 3的更新。")


In [ ]:
# 此处的切片只是fixture演示，不实现或覆盖M1的public_view。
reader_examples = []
for probe in M2_FIXTURE["probes"]:
    prefix = M2_FIXTURE["history"][:probe["end"]]
    project, attribute = m2_resolve_target(probe["question"], prefix)
    reader_examples.append({
        "t": probe["t"], "project": project, "attribute": attribute,
        "visible_ids": [m["id"] for m in prefix],
        "full_history_answer": rule_reader(probe["question"], prefix),
        "manual_gold": probe["gold"],
        "empty_context_answer": rule_reader(probe["question"], []),
    })
display(pd.DataFrame(reader_examples))
assert [r["full_history_answer"] for r in reader_examples] == ["V10", "V11", "V11", "V21"]


#### C · 先打包，再比较四臂

- recent两臂从最新消息向前选，遇到第一条放不下的消息就停止，以保留连续后缀；short最多检查3条。
- 检索臂按“相似度降序、平局时较新ID优先”保留最多8个候选，放不下的候选跳过，最后按ID升序返回。
- resolved臂只从公开历史识别对象、归一化同义词；`its`题优先打包最新focus，它可以在8个检索候选之外。
- 全部消息按整条计费，不截断正文；返回的ID必须唯一且升序。空历史或TF–IDF空词表返回空上下文。

最新focus若自身放不下，resolved臂返回空上下文，避免借用旧focus；不会把全历史消解出的对象偷偷传给回答器。


In [ ]:
def m2_validate_history(history):
    ids = [message["id"] for message in history]
    if any(type(message_id) is not int or message_id < 0 for message_id in ids):
        raise ValueError("消息id必须是非负整数。")
    if ids != sorted(set(ids)):
        raise ValueError("输入消息id必须唯一且按时间升序。")
    if any(not isinstance(message["text"], str) for message in history):
        raise ValueError("消息text必须是字符串。")


def m2_pack_messages(history, positions, budget, stop_on_fail=False):
    """positions是候选在history中的位置；返回消息副本，不修改输入。"""
    chosen, seen, used = [], set(), 0
    for position in positions:
        message = history[position]
        if message["id"] in seen:
            continue
        seen.add(message["id"])
        units = count_input_units(message["text"])
        if used + units <= budget:
            chosen.append(dict(message))
            used += units
        elif stop_on_fail:
            break
    return sorted(chosen, key=lambda message: message["id"])


In [ ]:
def m2_rank_tfidf(view, resolved=False):
    """返回检索诊断表；只在当前公开前缀拟合词表。"""
    history, question = view["history"], view["question"]
    columns = ["position", "id", "score", "query", "document"]
    if not history:
        return pd.DataFrame(columns=columns)
    documents = [message["text"] for message in history]
    if resolved:
        project, attribute = m2_resolve_target(question, history)
        question = " ".join(part for part in (project, attribute) if part)
        documents = [m2_normalize(text) for text in documents]
    # 使用和TfidfVectorizer默认设置相同的分词规则，显式处理全标点/空文本。
    vectorizer = TfidfVectorizer()
    analyzer = vectorizer.build_analyzer()
    if not any(analyzer(document) for document in documents):
        return pd.DataFrame(columns=columns)
    matrix = vectorizer.fit_transform(documents)
    scores = (matrix @ vectorizer.transform([question]).T).toarray().ravel()
    ranking = pd.DataFrame([
        {"position": i, "id": message["id"], "score": float(scores[i]),
         "query": question, "document": documents[i]}
        for i, message in enumerate(history)
    ])
    return ranking.sort_values(["score", "id"], ascending=[False, False]).reset_index(drop=True)


In [ ]:
def select_context(view, arm, cfg):
    """接口仅接受question/history；预算只约束消息正文。"""
    if set(view) != {"question", "history"}:
        raise ValueError("公开视图只能包含question/history。")
    if arm not in ARMS:
        raise ValueError(f"未知策略：{arm}")
    history = view["history"]
    m2_validate_history(history)
    budget = cfg["budgets"][arm]
    if type(budget) is not int or budget < 0:
        raise ValueError("上下文预算必须是非负整数。")
    if not history or budget == 0:
        return []
    if arm in ("recent_short", "recent_long"):
        positions = list(range(len(history) - 1, -1, -1))
        if arm == "recent_short":
            positions = positions[:3]
        return m2_pack_messages(history, positions, budget, stop_on_fail=True)

    resolved = arm == "resolved_tfidf"
    ranking = m2_rank_tfidf(view, resolved=resolved)
    positions = ranking["position"].head(8).tolist()
    if resolved and not M2_PROJECT.search(view["question"]) and re.search(r"\bits\b", view["question"], re.I):
        anchors = [i for i, message in enumerate(history) if M2_FOCUS.search(message["text"])]
        if anchors:
            anchor = anchors[-1]
            if count_input_units(history[anchor]["text"]) > budget:
                return []
            positions = [anchor] + [i for i in positions if i != anchor]
    return m2_pack_messages(history, positions, budget)


In [ ]:
# 观察t=3：short虽然拿到了language事实(id 6)，却丢失了focus(id 5)。
M2_DEMO_PROBE = M2_FIXTURE["probes"][3]
M2_DEMO_VIEW = {
    "question": M2_DEMO_PROBE["question"],
    "history": deepcopy(M2_FIXTURE["history"][:M2_DEMO_PROBE["end"]]),
}
for arm in ARMS:
    context = select_context(M2_DEMO_VIEW, arm, M2_CFG)
    units = count_input_units(m2_context_text(context))
    print(f"{arm}: context={units}/{M2_CFG['budgets'][arm]}, "
          f"answer={rule_reader(M2_DEMO_VIEW['question'], context)}")
    display(pd.DataFrame(context, columns=["id", "text"]))

print("原问题检索排序（前8名）：")
display(m2_rank_tfidf(M2_DEMO_VIEW).head(8).round({"score": 4}))
print("公开历史消解后的检索排序（前8名；另优先打包focus）：")
display(m2_rank_tfidf(M2_DEMO_VIEW, resolved=True).head(8).round({"score": 4}))


#### D · 评价器读取真值，策略与回答器只读取公开输入

`evaluate_episode(episode, cfg)`为每题每臂返回一行，字段与输出契约一致。
当前M1的`public_view`仍是占位函数，所以本里程碑由评价器显式构造同契约的公开前缀；待联调时与M1实现交叉核对。
这里定义 `end` 为右开切片边界，不从隐藏标签推断策略。

`coverage`检查evidence的消息ID是否入选；`outdated`检查回答是否属于old_values；
`y`对照gold；`utility = y - cfg["lambda"] * cost / cfg["cost_reference"]`。
`construction_ms`只测上下文构造，受本机运行状态影响，不要求重复运行完全一致。


In [ ]:
M2_OUTCOME_COLUMNS = [
    "seed", "scenario", "t", "arm", "question", "y", "cost", "utility",
    "coverage", "outdated", "construction_ms", "qtype", "state",
]


def m2_public_prefix(episode, probe):
    """M1未接通时的显式边界适配器，不替换M1.public_view。"""
    end = probe["end"]
    if type(end) is not int or not 0 <= end <= len(episode["history"]):
        raise ValueError("end必须是当前history范围内的右开切片边界。")
    return {"question": probe["question"],
            "history": [{"id": m["id"], "text": m["text"]}
                        for m in episode["history"][:end]]}


def evaluate_episode(episode, cfg):
    """真值只在策略选择和回答之后用于评分；不自动导出或宣称审计通过。"""
    m2_validate_history(episode["history"])
    rows = []
    for probe in episode["probes"]:
        view = m2_public_prefix(episode, probe)
        for arm in ARMS:
            start = perf_counter()
            context = select_context(view, arm, cfg)
            elapsed_ms = (perf_counter() - start) * 1000
            ids = [message["id"] for message in context]
            if ids != sorted(set(ids)) or not set(ids) <= {m["id"] for m in view["history"]}:
                raise ValueError("策略返回了重复、乱序或非公开消息。")
            if count_input_units(m2_context_text(context)) > cfg["budgets"][arm]:
                raise ValueError("上下文超过预算。")
            answer = rule_reader(view["question"], context)
            cost = count_input_units(m2_build_prompt(view["question"], context))
            correct = int(answer == probe["gold"])
            rows.append({
                "seed": episode["seed"], "scenario": episode["scenario"], "t": probe["t"],
                "arm": arm, "question": probe["question"], "y": correct, "cost": cost,
                "utility": correct - cfg["lambda"] * cost / cfg["cost_reference"],
                "coverage": int(probe["evidence"] in ids),
                "outdated": int(answer in probe["old_values"]),
                "construction_ms": elapsed_ms, "qtype": probe["qtype"], "state": probe["state"],
            })
    result = pd.DataFrame(rows, columns=M2_OUTCOME_COLUMNS)
    if result.duplicated(["seed", "t", "arm"]).any():
        raise ValueError("四臂表seed/t/arm不唯一。")
    return result


In [ ]:
def m2_inspect_probe(episode, probe, cfg):
    """演示用额外诊断列，和交给M3的固定schema分开。"""
    view = m2_public_prefix(episode, probe)
    rows = []
    for arm in ARMS:
        context = select_context(view, arm, cfg)
        parts = {
            "instruction_units": count_input_units(M2_INSTRUCTION),
            "format_units": count_input_units("Context:\nQuestion: "),
            "context_units": count_input_units(m2_context_text(context)),
            "question_units": count_input_units(view["question"]),
        }
        cost = count_input_units(m2_build_prompt(view["question"], context))
        assert sum(parts.values()) == cost
        rows.append({
            "t": probe["t"], "arm": arm,
            "selected_ids": [message["id"] for message in context],
            "answer": rule_reader(view["question"], context), "gold": probe["gold"],
            "evidence": probe["evidence"], "budget": cfg["budgets"][arm],
            **parts, "cost": cost,
        })
    return pd.DataFrame(rows)


M2_FIXTURE_DETAILS = pd.concat([
    m2_inspect_probe(M2_FIXTURE, probe, M2_CFG) for probe in M2_FIXTURE["probes"]
], ignore_index=True)
display(M2_FIXTURE_DETAILS)

demo_context = select_context(M2_DEMO_VIEW, "recent_short", M2_CFG)
print("t=3 / recent_short 的完整输入，可对照上表逐项数成本：")
print(m2_build_prompt(M2_DEMO_VIEW["question"], demo_context))


In [ ]:
M2_FIXTURE_OUTCOMES = evaluate_episode(M2_FIXTURE, M2_CFG)
display(M2_FIXTURE_OUTCOMES.round({"utility": 6, "construction_ms": 3}))

# 将固定接口表与手工核对信息并排展示；不把诊断列混入M3的接口表。
comparison = M2_FIXTURE_OUTCOMES.merge(
    M2_FIXTURE_DETAILS[["t", "arm", "selected_ids", "answer", "gold", "context_units"]],
    on=["t", "arm"], validate="one_to_one",
)
display(comparison[[
    "t", "arm", "selected_ids", "answer", "gold", "y", "coverage", "outdated", "cost", "utility"
]].round({"utility": 6}))
print("fixture共4题 × 4臂 =", len(M2_FIXTURE_OUTCOMES), "行；未生成真实开发集或人工审计结论。")


#### E · 用可定位的边界案例验证性质

下面的检查不估计总体性能；它们针对具体实现风险。
预算测试使用刻意缩小的配置副本，不改公共设计。未来隔离测试保持当前公开前缀不变，仅改未来消息；
元数据隔离测试单独改变gold/隐藏标签，确认评价器传给选择器的公开输入不变。


In [ ]:
M2_CHECKS = []

# 1. 手写预期：short在t=2找不到P1事实，在t=3找不到its的focus。
short = M2_FIXTURE_OUTCOMES.query("arm == 'recent_short'").sort_values("t")
assert short["y"].tolist() == [1, 1, 0, 0]
assert short["coverage"].tolist() == [1, 1, 0, 1]
assert M2_FIXTURE_DETAILS.query("arm == 'recent_short'")["answer"].tolist() == [
    "V10", "V11", "UNKNOWN", "UNKNOWN"
]
assert len(M2_FIXTURE_OUTCOMES) == 16
assert M2_FIXTURE_OUTCOMES.columns.tolist() == M2_OUTCOME_COLUMNS
assert not M2_FIXTURE_OUTCOMES.duplicated(["seed", "t", "arm"]).any()
M2_CHECKS.append({"case": "手写答案/coverage/16行唯一键", "result": "pass"})

# 2. 恰好预算3、超出预算2；长窗遇到超长末条就停，检索打包允许跳过。
tiny = [{"id": 0, "text": "a b"}, {"id": 1, "text": "c"}]
tiny_cfg = deepcopy(M2_CFG)
tiny_cfg["budgets"]["recent_long"] = 3
assert [m["id"] for m in select_context({"question": "?", "history": tiny}, "recent_long", tiny_cfg)] == [0, 1]
tiny_cfg["budgets"]["recent_long"] = 2
assert [m["id"] for m in select_context({"question": "?", "history": tiny}, "recent_long", tiny_cfg)] == [1]
oversize = [{"id": 0, "text": "a"}, {"id": 1, "text": "b c d"}]
assert m2_pack_messages(oversize, [1, 0], 2, stop_on_fail=True) == []
assert [m["id"] for m in m2_pack_messages(oversize, [1, 0], 2)] == [0]
assert [m["id"] for m in m2_pack_messages(tiny, [1, 1, 0], 3)] == [0, 1]
M2_CHECKS.append({"case": "预算边界/整条打包/重复候选", "result": "pass"})

# 3. 空历史、全标点词表、无focus；检查所有臂的输出约束。
for arm in ARMS:
    assert select_context({"question": "What is its dataset?", "history": []}, arm, M2_CFG) == []
    for probe in M2_FIXTURE["probes"]:
        view = m2_public_prefix(M2_FIXTURE, probe)
        context = select_context(view, arm, M2_CFG)
        ids = [m["id"] for m in context]
        assert ids == sorted(set(ids)) and all(i < probe["end"] for i in ids)
        assert count_input_units(m2_context_text(context)) <= M2_CFG["budgets"][arm]
punctuation = {"question": "?", "history": [{"id": 0, "text": "..."}]}
assert select_context(punctuation, "tfidf", M2_CFG) == []
assert select_context(punctuation, "resolved_tfidf", M2_CFG) == []
assert rule_reader("What is its dataset?", [M2_FIXTURE["history"][3]]) == "UNKNOWN"
M2_CHECKS.append({"case": "空历史/空词表/缺focus/预算和时序", "result": "pass"})


In [ ]:
# 4. 新旧版本和复述由手写常量验证，而不是拿回答器证明自己的输出。
question = "What is P1 current dataset?"
history = M2_FIXTURE["history"]
assert rule_reader(question, [history[0]]) == "V10"
assert rule_reader(question, [history[0], history[3]]) == "V11"
assert rule_reader(question, [history[3], history[8]]) == "V11"
assert rule_reader(question, []) == "UNKNOWN"
assert rule_reader("What is P1 current due date?", history) == "UNKNOWN"
# 合法的两消息历史：新事实过长、超出所有臂预算；检索跳过它而返回旧值。
old_episode = {
    "seed": 43, "scenario": "fixture",
    "history": [dict(history[0]), {"id": 1, "text": history[3]["text"] + " extra" * 600}],
    "probes": [dict(M2_FIXTURE["probes"][2], t=0, end=2, evidence=1)],
}
old_rows = evaluate_episode(old_episode, M2_CFG)
assert old_rows["y"].tolist() == [0, 0, 0, 0]
assert old_rows["outdated"].tolist() == [0, 0, 1, 1]
M2_CHECKS.append({"case": "旧值/更新/同值复述/UNKNOWN/outdated评分", "result": "pass"})

# 5. 同一公开前缀，未来事实不同，当前结果除耗时外完全不变。
before = deepcopy(M2_FIXTURE)
before["probes"] = [before["probes"][1]]
changed_future = deepcopy(before)
changed_future["history"][6]["text"] = "For project P1: the dataset is V999."
changed_future["history"].append({"id": 9, "text": "Focus is project P999."})
pd.testing.assert_frame_equal(
    evaluate_episode(before, M2_CFG).drop(columns="construction_ms"),
    evaluate_episode(changed_future, M2_CFG).drop(columns="construction_ms"),
)
changed_labels = deepcopy(before)
changed_labels["probes"][0].update(gold="V999", qtype="hidden_changed", state="hidden_changed")
assert m2_public_prefix(before, before["probes"][0]) == m2_public_prefix(
    changed_labels, changed_labels["probes"][0]
)
assert before == dict(deepcopy(M2_FIXTURE), probes=[deepcopy(M2_FIXTURE["probes"][1])])
M2_CHECKS.append({"case": "未来隔离/隐藏标签隔离/输入未被修改", "result": "pass"})

# 6. 最新focus自身超预算时，resolved不带着旧focus去回答。
focus_view = {"question": "What is its dataset?", "history": [
    {"id": 0, "text": "Focus is project P1."},
    {"id": 1, "text": "For project P1: the dataset is V10."},
    {"id": 2, "text": "Focus is project P2. " + "extra " * 30},
]}
small_cfg = deepcopy(M2_CFG)
small_cfg["budgets"]["resolved_tfidf"] = 20
assert select_context(focus_view, "resolved_tfidf", small_cfg) == []
assert count_input_units("P1: V11.") == 4
np.testing.assert_allclose(
    M2_FIXTURE_OUTCOMES["utility"],
    M2_FIXTURE_OUTCOMES["y"] - 0.5 * M2_FIXTURE_OUTCOMES["cost"] / 512,
)
M2_CHECKS.append({"case": "focus锚点预算/正则计数/效用公式", "result": "pass"})

# 7. 十条完全同分的消息应优先取最新8条，最终恢复时间顺序。
tie_view = {"question": "dataset", "history": [
    {"id": i, "text": "dataset"} for i in range(10)
]}
assert [m["id"] for m in select_context(tie_view, "tfidf", M2_CFG)] == list(range(2, 10))
for bad_view in [
    dict(M2_DEMO_VIEW, gold="V21"),
    {"question": "?", "history": [{"id": 0, "text": "a"}, {"id": 0, "text": "b"}]},
]:
    try:
        select_context(bad_view, "tfidf", M2_CFG)
    except ValueError:
        pass
    else:
        raise AssertionError("隐藏字段或重复ID没有被拒绝。")
M2_CHECKS.append({"case": "检索同分/8候选上限/非法公开输入", "result": "pass"})
display(pd.DataFrame(M2_CHECKS))


In [ ]:
def export_strategy_evidence(episodes, outcomes, run_dir):
    """后续里程碑：真实开发集、10题审计与策略检查导出。"""
    raise NotImplementedError("M2后续任务：M1联调后导出真实开发证据；本节fixture不冒充开发集。")


### M2 本里程碑的可核对观察与下一步

- t=0只允许看到id 0–2，因此P1的答案是V10；id 3的V11属于未来。
- t=1的最近三条为id 2、3、4，既有focus也有更新事实，可回答V11。
- t=2的最近三条为id 5、6、7，不含P1事实，应返回UNKNOWN。
- t=3的最近三条为id 6、7、8：最新language证据id 6已入选（coverage=1），
  但最新focus id 5不在上下文，its无法消解，所以y=0。这是coverage不等于accuracy的一个具体反例。
- 成本表把共同指令、格式标签、上下文与问题分列，四项之和等于cost；未知回答仍然有输入成本，效用可以为负。

以上仅是手工fixture上的机制核对，不证明某个策略在开发集或正式实验上更好。
本里程碑未完成M1模板联调、真实开发集、十题双人审计、完整导出和M3/M4验收。
下一步与M1确认英文模板/同义词、`end`右开边界和`evidence`语义，再交付单对话四臂表给M3。
M2完整研究记录仍需在 `deliverables/M2_notes.md` 补充自己的理解、引用、修改和联调证据。

本地运行后可看到全部中间表；提交Git前使用 `scripts/clean_notebook.py` 清输出，再运行结构检查。


<a id="member-m3"></a>
## M3 · 贝叶斯建模与回放
**后验更新、决策、优化、回放与数值验证由你实现。任务区间m3-*。**
输入：问题、已知四候选cost、M2单对话评价表；输出：统一学习器、回放、推导与核查；M4接收。

1. **先推导。** 从Bernoulli似然/Beta先验推导TS后验，手算收到1、1、0后的参数与均值；解释成本为什么不应计为失败次数。
2. **实现TS与参照。** 定义状态、采样和成本决策；实现四固定臂与基于公开问题的RuleRouter，研究先验和探索。
3. **研究CTS。** 每臂贝叶斯logistic与高斯先验；自己推导MAP目标、梯度、Hessian、Laplace协方差。实现优化、采样、每5次已选反馈重拟合；解释特征共线性与数值稳定性。
4. **实现回放。** 每新对话新建学习器；choose只看q/costs，update只看已选臂0/1反馈；全臂表仅属于评估器。
5. **验证交付。** 自己写手算一致、有限差分、协方差、重置、反馈隔离案例；报告拟合失败，不静默吞掉。交M4微型输入/输出及M2真实表上的检查。

**研究问题：** 如何检测未选奖励泄漏？同seed是否复现？特征如何不借用隐藏标签？TS与CTS差异是否足以构成严格特征消融？
**输出：** audit/method_checks.json、tables/development_replay.csv；推导和局限写M3_notes。


In [ ]:
def question_features(question):
    """M3实现：三维[截距, 显式项目名, its]，只读取问题。"""
    raise NotImplementedError("M3：实现公开特征")

class Selector:
    def __init__(self, method, seed, cfg):
        """M3实现：支持固定臂、TS、LaplaceCTS、RuleRouter，初始化状态。"""
        raise NotImplementedError("M3：设计学习器状态")

    def choose(self, question, costs):
        """M3实现：返回ARMS索引，不可读取当前真实奖励。"""
        raise NotImplementedError("M3：实现决策")

    def update(self, arm_index, reward, question):
        """M3实现：reward仅为已选臂0/1正确性反馈。"""
        raise NotImplementedError("M3：实现后验更新")

def replay_episode(arm_outcomes, method, algorithm_seed, cfg):
    """M3实现：单对话全臂表→该方法逐题回放DataFrame。"""
    raise NotImplementedError("M3：实现信息隔离的回放")


In [ ]:
# TODO M3：Beta手算对照、MAP梯度/Hessian有限差分。
# 第5次反馈前后、协方差有效性、新对话重置。
# 设计能暴露未选奖励泄漏的测试，而非只检查函数签名。
# 用自建小表再用M2交付表；保留失败数值与修复说明。


### M3研究记录（本人填写）
推导如何对应自己的代码？拟合失败如何处理？哪些是测试性质、哪些是建模假设？什么条件下方法会失效？
M4接收的接口、检查证据和方法说明：待填写。不要预写CTS更好。


<a id="member-m4"></a>
## M4 · 集成、实验设计与统计
**你要编写调度、开发选择、冻结、统计和归档。任务区间m4-*。**
输入：M1–M3实现、证据、审查；输出：真实run、表、manifest和判断；M3复核、M5接收。

1. **先做统计手算。** 自建少量对话小表验证配对和算法seed的对话内平均；这是fixture，不能作为研究结果。
2. **接通demo。** 编写入口逐模块调用；按场景仅用开发平均效用选固定基线，说明平局规则，不用测试结果选基线。
3. **实现冻结与失败管理。** 核验M1/M2审计、M3方法检查、全组确认；保存配置/源码/证据快照、哈希、版本，检查seed分离、防覆盖；登记每次正式尝试和已看数据情况。
4. **正式调度。** 每场景30条×300题，四臂评价与方法回放；新对话重置，两个算法seed不是两份独立数据。先查完整性再看效果。
5. **自行实现统计与判断。** 对话级配对bootstrap，主次比较、95%及两主比较97.5%区间；区间含0不等于等效，点估计到0.02不等于下界超过0.02。
6. **可复现交付。** 导出同run完整表和manifest，让M3复核聚合/配对/区间，M5独立读取。保留负结果，不换seed直到显著。

**输出：** data/evaluation_dialogues.jsonl.gz；tables/evaluation_outcomes.csv、replay.csv、episode_metrics.csv、primary_effects.csv、secondary_effects.csv、method_summary.csv；config/frozen_config.json、notebook_snapshot.ipynb；audit/review_snapshot.json；reports/M4_handoff.md；根级manifest.json。实现与产物均待你完成。


In [ ]:
def choose_fixed_baselines(development_outcomes):
    """M4实现：只用开发平均效用，记录平局规则。"""
    raise NotImplementedError("M4：实现开发选择")

def freeze_protocol(cfg, baselines, evidence_paths, run_dir):
    """M4实现：核验交接、保存冻结快照/seed记录，在测试数据生成前调用。"""
    raise NotImplementedError("M4：实现冻结与防覆盖")

def run_evaluation(episodes, methods, algorithm_seeds, cfg):
    """M4实现：调用M2/M3，返回全臂表/回放表，核验规模与唯一键。"""
    raise NotImplementedError("M4：实现实验集成")

def aggregate_dialogues(replay_table):
    """M4实现：对话级指标；算法seed不视为新增独立样本。"""
    raise NotImplementedError("M4：实现正确聚合")

def paired_effects(episode_metrics, baselines, cfg):
    """M4实现：配对重采样独立对话，输出主次效应表。"""
    raise NotImplementedError("M4：实现bootstrap与判断")

def export_run(run_dir, artifacts, metadata):
    """M4实现：按契约保存结果/manifest供M5独立验证。"""
    raise NotImplementedError("M4：实现归档")


In [ ]:
# TODO M4：先写手算对照，再实现调度、配对与聚合。
# 检查dev/test seed不相交、轨迹无缺失/重复、算法seed齐全。
# 研究错误地以题为单位重采样会有何影响。
# 接口验收后自己增加development/demo入口；正式入口先核验冻结。
# 未满足条件明确停止，不填假结果，不自动替成员签名。


### M4结果判断（实际实验后本人填写）
主比较、基线选择依据、统计单位和n；效应方向/区间/0.02含义；负结果、异常、修正与已看情况；M3复核；交付run/commit/manifest校验和：待填写。未完成前只写计划。


<a id="member-m5"></a>
## M5 · 可视化研究、结果解释与汇报
**图表代码、读取校验与评价由你完成。任务区间m5-*。**
输入：M4完整归档及M1–M3说明；输出：自己设计的图、解读与汇报；全组复核。

1. **不等结果开始。** 设计叙事与每图的问题，用明确标为fixture的小表探索图型/颜色/误差线。最终不能混入假数据。
2. **实现独立读取。** 指定run路径，校验manifest、文件、模式、配置和规模，不依赖M4内存，不自动取“最新目录”；和M4共同验收契约。
3. **自主设计与编码。** 下表是问题菜单，不是画法答案。解释图型、聚合和区间选择；可协商合并或调整图数，不为凑数重复展示。
4. **自己评估。** 对照CSV复核数值，每图写观察→证据→局限，与对应成员核对。不显著或方向相反也如实展示，不能复制参考本结论。
5. **完成汇报。** M1背景、M2策略、M3方法、M4统计、M5叙事；旧计划约15–16页slides/15–20分钟视频，须核对课程要求，说明个人贡献与AI辅助。

| 建议问题 | 需要的数据 | 联合复核 |
|---|---|---|
| 主效应是否明确？ | primary/secondary_effects | M4/M3 |
| 准确率、成本、效用如何取舍？ | episode_metrics/method_summary | M2 |
| 效用差由什么组成？ | 同一基线的分量差 | M2/M4 |
| 不同需求怎样选臂？ | replay及题型 | M3/M1 |
| 收益是否贯穿对话？ | 配对逐题轨迹 | M4 |
| 策略有何互补/失败机制？ | 全臂表与具体例子 | M1/M2 |
| 突变前后是否相同？ | 阶段对话配对表 | M4 |

**边界：** 描述性区间不增加主检验；逐点带不是同时置信带；固定基线累计差不是理论regret；proxy成本不是实际费用。
**输出：** figures/<figure_id>.png/.svg、tables/figure_<figure_id>.csv、reports/figure_interpretations.md、reports/figure_manifest.json。只新增图表文件，不改M4数据；重绘使用新figure_id或明确render子目录并记录来源。


In [ ]:
def load_verified_run(run_dir):
    """M5实现：独立校验归档后读取绘图所需数据。"""
    raise NotImplementedError("M5：实现来源校验和读取")

def build_figures(results, cfg):
    """M5实现：自行选择图型、聚合/误差线，返回图对象和对应数据表。"""
    raise NotImplementedError("M5：设计并实现可视化")

def export_visual_evidence(figures, figure_tables, source_run, render_id):
    """M5实现：保存图/表/来源清单，不修改上游原始数据。"""
    raise NotImplementedError("M5：实现图表交付")


In [ ]:
# TODO M5：用fixture先检查绘图；模拟缺文件/改CSV/混run验证读取器。
# 新内核只执行共享单元与M5模块，不依赖M4变量。
# 检查数据-图一致、单位/区间、负值标签、图例。
# 实现后增加绘图调用；本人写解读，不用固定文字自动宣布胜出。


### M5逐图研究记录（本人填写）
每图的问题、来源run与表、汇总方式、观察、支持/不支持的结论、局限、复核人：待填写。
随后补slides、分工、视频链接与提交清单。未有结果时只写计划与示意图设计。


## 完成标准
todo → doing → review → done：自己的实现与研究说明完成，自己设计的案例通过，接收人能按接口使用并复核后再标done。结构检查不证明算法正确或项目完成。

正式评估前冻结共同协议；看过测试结果之后的探索、修正和失败必须留记录，不能反复换seed至显著。

课程线索：Lecture 2的Beta/MAP/Laplace，Lecture 6的bandit/后验采样，Lecture 4的开发/评估隔离，Lecture 1的建模与不确定性。成员自行阅读原课件、推导和引用，本工作本不提供完成的推导答案。

模块完成后由M4自行编写集成入口；各成员用自己的测试、实验与交叉复核证明成果。
